# Model Evaluation Comparison: YOLOv5 vs YOLOv8

Unified model loading and evaluation framework for comparing trained detection models.

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import yaml
import glob
import time
import psutil
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict

In [4]:
# Unified Model Loader
class ModelLoader:
    """Unified model loader for YOLOv5 and YOLOv8 models."""
    
    MODEL_TYPE_MAP = {
        'v5': 'yolov5',
        'v8': 'yolov8'
    }
    
    def __init__(self, model_path, device='cuda'):
        self.model_path = model_path
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.model = None
        self.model_type = self._detect_model_type()
        
    def _detect_model_type(self):
        """Detect model type from filename."""
        path_lower = self.model_path.lower()
        if 'v8' in path_lower or 'yolov8' in path_lower:
            return 'yolov8'
        elif 'v5' in path_lower or 'yolov5' in path_lower:
            return 'yolov5'
        else:
            # Default to YOLOv8 for generic .pt files
            return 'yolov8'
    
    def load(self):
        """Load model based on detected type."""
        if not os.path.exists(self.model_path):
            raise FileNotFoundError(f"Model weights not found: {self.model_path}")
        
        # Use Ultralytics API for both YOLOv5 and YOLOv8 for consistency
        self.model = YOLO(self.model_path)
        
        return self.model
    
    def predict(self, image):
        """Run inference uniformly across model types."""
        if self.model is None:
            self.load()
        
        return self.model.predict(image, verbose=False)
    
    def validate(self, data_yaml, img_size):
        """Run validation uniformly across model types."""
        if self.model is None:
            self.load()
        
        return self.model.val(data=data_yaml, imgsz=img_size, device=self.device)
    
    def get_type(self):
        """Return model type."""
        return self.model_type
    
    def __del__(self):
        """Cleanup on deletion."""
        if self.model is not None:
            del self.model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [5]:
# Configuration
CONFIG = {
    'data_yaml': 'vehicle_dataset/data.yaml',
    'img_size': 640,
    'yolov5_weights': 'models/yolov5_1k.pt',
    'yolov8_weights': 'models/yolo.pt',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"Device: {CONFIG['device']}")

Device: cpu


In [6]:
def evaluate_model(model_loader, data_yaml, img_size):
    """
    Unified model evaluation for YOLOv5 and YOLOv8.
    Returns dict with mAP@0.5, mAP@0.5:0.95, precision, recall.
    """
    try:
        results = model_loader.validate(data_yaml, img_size)
        
        metrics = {
            'mAP@0.5': results.box.map50,
            'mAP@0.5:0.95': results.box.map,
            'Precision': results.box.mp,
            'Recall': results.box.mr
        }
        return metrics
    except Exception as e:
        print(f"Error evaluating {model_loader.get_type().upper()} model: {e}")
        return {'mAP@0.5': 0, 'mAP@0.5:0.95': 0, 'Precision': 0, 'Recall': 0}

In [7]:
# Note: evaluate_yolov8 consolidated into unified evaluate_model() above
pass

In [8]:
def benchmark_inference(model_loader, image_paths, warmup_runs=5, num_runs=20):
    """
    Benchmark inference speed on given images.
    Returns average latency (ms/image) and FPS.
    """
    # Warmup runs
    for i in range(min(warmup_runs, len(image_paths))):
        img = cv2.imread(image_paths[i % len(image_paths)])
        if img is None:
            continue
        model_loader.predict(img)
    
    # Actual benchmark
    total_time = 0
    num_inferences = 0
    
    for i in range(num_runs):
        img_path = image_paths[i % len(image_paths)]
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        start = time.time()
        model_loader.predict(img)
        total_time += (time.time() - start)
        num_inferences += 1
    
    avg_latency = (total_time / num_inferences) * 1000 if num_inferences > 0 else 0
    fps = 1000 / avg_latency if avg_latency > 0 else 0
    
    return avg_latency, fps

In [9]:
def get_model_size(weights_path):
    """
    Get model file size in MB.
    """
    if os.path.exists(weights_path):
        size_mb = os.path.getsize(weights_path) / (1024 * 1024)
        return size_mb
    return 0

In [10]:
def measure_resource_usage(model_loader, image_paths, num_runs=20):
    """
    Measure CPU, RAM, and GPU usage during inference.
    Returns dict with avg CPU %, RAM (MB), GPU memory (MB).
    """
    process = psutil.Process(os.getpid())
    
    # Track metrics
    cpu_usage = []
    ram_usage = []
    gpu_memory = []
    
    # Clear GPU cache before measurement
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    
    for i in range(num_runs):
        img_path = image_paths[i % len(image_paths)]
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # Measure before
        cpu_percent = process.cpu_percent(interval=0.01)
        ram_mb = process.memory_info().rss / (1024 * 1024)
        
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            gpu_mem_before = torch.cuda.memory_allocated() / (1024 * 1024)
        
        # Run inference
        model_loader.predict(img)
        
        # Measure after
        cpu_usage.append(cpu_percent)
        ram_usage.append(ram_mb)
        
        if torch.cuda.is_available():
            gpu_mem_after = torch.cuda.max_memory_allocated() / (1024 * 1024)
            gpu_memory.append(gpu_mem_after)
    
    metrics = {
        'CPU (%)': np.mean(cpu_usage) if cpu_usage else 0,
        'RAM (MB)': np.mean(ram_usage) if ram_usage else 0,
        'GPU Memory (MB)': np.mean(gpu_memory) if gpu_memory else 0
    }
    
    return metrics

In [11]:
def load_test_images(data_yaml, max_images=50):
    """
    Load test images from dataset.
    """
    with open(data_yaml, 'r') as f:
        data = yaml.safe_load(f)
    
    # Get test images path
    test_path = data.get('path', '')
    test_dir = os.path.join(test_path, data.get('test', 'images/test'))
    
    image_paths = glob.glob(os.path.join(test_dir, '*.jpg')) + \
                  glob.glob(os.path.join(test_dir, '*.png')) + \
                  glob.glob(os.path.join(test_dir, '*.jpeg'))
    
    return sorted(image_paths)[:max_images]

In [12]:
# Load test images for benchmarking
print("Loading test images...")
test_images = load_test_images(CONFIG['data_yaml'])
print(f"Found {len(test_images)} test images")

Loading test images...
Found 50 test images


In [13]:
# Load and evaluate YOLOv5
print("\n=== Evaluating YOLOv5 ===")
print(f"Weights: {CONFIG['yolov5_weights']}")

yolov5_loader = ModelLoader(CONFIG['yolov5_weights'], CONFIG['device'])
print(f"Loading {yolov5_loader.get_type().upper()} model...")
yolov5_loader.load()

print("Running validation...")
yolov5_metrics = evaluate_model(yolov5_loader, CONFIG['data_yaml'], CONFIG['img_size'])
print(f"YOLOv5 Metrics: {yolov5_metrics}")

print("\nBenchmarking YOLOv5 inference...")
yolov5_latency, yolov5_fps = benchmark_inference(yolov5_loader, test_images)
print(f"YOLOv5 Avg Latency: {yolov5_latency:.2f} ms/image")
print(f"YOLOv5 FPS: {yolov5_fps:.2f}")

print("\nMeasuring YOLOv5 resource usage...")
yolov5_resources = measure_resource_usage(yolov5_loader, test_images)
print(f"YOLOv5 Resources: CPU={yolov5_resources['CPU (%)']:.2f}%, RAM={yolov5_resources['RAM (MB)']:.2f}MB, GPU={yolov5_resources['GPU Memory (MB)']:.2f}MB")

yolov5_size = get_model_size(CONFIG['yolov5_weights'])
print(f"YOLOv5 Model Size: {yolov5_size:.2f} MB")


=== Evaluating YOLOv5 ===
Weights: models/yolov5_1k.pt
Loading YOLOV5 model...
Running validation...
Ultralytics 8.4.34 🚀 Python-3.11.15 torch-2.2.2 CPU (Apple M4 Pro)


YOLOv5s summary (fused): 85 layers, 9,111,923 parameters, 0 gradients, 23.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 214.2±43.5 MB/s, size: 70.4 KB)
val: Scanning /Users/biney/Documents/school/year3/term2/foundation_data_sceience/final_project/object-detection-for-traffic-using-yolo/vehicle_dataset/labels/test.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 484.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 2.6s/it 4:042.6ss
                   all       1500      13607      0.974      0.966      0.991      0.866
Speed: 0.1ms preprocess, 159.7ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /Users/biney/Documents/school/year3/term2/foundation_data_sceience/final_project/object-detection-for-traffic-using-yolo/runs/detect/val5
YOLOv5 Metrics: {'mAP@0.5': 0.991479327144846, 'mAP@0.5:0.95': 0.8656406561856838, 'Precision': 0.973924082525212, 'Reca

In [14]:
# Load and evaluate YOLOv8
print("\n=== Evaluating YOLOv8 ===")
print(f"Weights: {CONFIG['yolov8_weights']}")

yolov8_loader = ModelLoader(CONFIG['yolov8_weights'], CONFIG['device'])
print(f"Loading {yolov8_loader.get_type().upper()} model...")
yolov8_loader.load()

print("Running validation...")
yolov8_metrics = evaluate_model(yolov8_loader, CONFIG['data_yaml'], CONFIG['img_size'])
print(f"YOLOv8 Metrics: {yolov8_metrics}")

print("\nBenchmarking YOLOv8 inference...")
yolov8_latency, yolov8_fps = benchmark_inference(yolov8_loader, test_images)
print(f"YOLOv8 Avg Latency: {yolov8_latency:.2f} ms/image")
print(f"YOLOv8 FPS: {yolov8_fps:.2f}")

print("\nMeasuring YOLOv8 resource usage...")
yolov8_resources = measure_resource_usage(yolov8_loader, test_images)
print(f"YOLOv8 Resources: CPU={yolov8_resources['CPU (%)']:.2f}%, RAM={yolov8_resources['RAM (MB)']:.2f}MB, GPU={yolov8_resources['GPU Memory (MB)']:.2f}MB")

yolov8_size = get_model_size(CONFIG['yolov8_weights'])
print(f"YOLOv8 Model Size: {yolov8_size:.2f} MB")


=== Evaluating YOLOv8 ===
Weights: models/yolo.pt
Loading YOLOV8 model...
Running validation...
Ultralytics 8.4.34 🚀 Python-3.11.15 torch-2.2.2 CPU (Apple M4 Pro)


Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1172.9±1294.6 MB/s, size: 83.0 KB)
val: Scanning /Users/biney/Documents/school/year3/term2/foundation_data_sceience/final_project/object-detection-for-traffic-using-yolo/vehicle_dataset/labels/test.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 786.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 2.6s/it 4:052.7ss
                   all       1500      13607      0.986      0.983      0.994      0.909
Speed: 0.1ms preprocess, 160.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /Users/biney/Documents/school/year3/term2/foundation_data_sceience/final_project/object-detection-for-traffic-using-yolo/runs/detect/val6
YOLOv8 Metrics: {'mAP@0.5': 0.994325738468458, 'mAP@0.5:0.95': 0.9090937223052192, 'Precision': 0.9862154645271904, 'R

In [15]:
# Create comparison DataFrame
comparison_data = {
    'Model': ['YOLOv5', 'YOLOv8'],
    'mAP@0.5': [yolov5_metrics['mAP@0.5'], yolov8_metrics['mAP@0.5']],
    'mAP@0.5:0.95': [yolov5_metrics['mAP@0.5:0.95'], yolov8_metrics['mAP@0.5:0.95']],
    'Precision': [yolov5_metrics['Precision'], yolov8_metrics['Precision']],
    'Recall': [yolov5_metrics['Recall'], yolov8_metrics['Recall']],
    'FPS': [yolov5_fps, yolov8_fps],
    'Inference Time (ms)': [yolov5_latency, yolov8_latency],
    'Model Size (MB)': [yolov5_size, yolov8_size],
    'CPU Usage (%)': [yolov5_resources['CPU (%)'], yolov8_resources['CPU (%)']],
    'RAM Usage (MB)': [yolov5_resources['RAM (MB)'], yolov8_resources['RAM (MB)']],
    'GPU Memory (MB)': [yolov5_resources['GPU Memory (MB)'], yolov8_resources['GPU Memory (MB)']]
}

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("MODEL EVALUATION COMPARISON")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)


MODEL EVALUATION COMPARISON
 Model  mAP@0.5  mAP@0.5:0.95  Precision   Recall       FPS  Inference Time (ms)  Model Size (MB)  CPU Usage (%)  RAM Usage (MB)  GPU Memory (MB)
YOLOv5 0.991479      0.865641   0.973924 0.966197 49.066976            20.380306        17.638308         698.70      1578.06250                0
YOLOv8 0.994326      0.909094   0.986215 0.983239 46.144852            21.670890        85.352084         698.15      1795.96875                0


In [16]:
# Calculate performance differences
print("\n=== Performance Differences ===")
print(f"mAP@0.5 Difference (YOLOv8 - YOLOv5): {yolov8_metrics['mAP@0.5'] - yolov5_metrics['mAP@0.5']:.4f}")
print(f"mAP@0.5:0.95 Difference (YOLOv8 - YOLOv5): {yolov8_metrics['mAP@0.5:0.95'] - yolov5_metrics['mAP@0.5:0.95']:.4f}")
print(f"Precision Difference (YOLOv8 - YOLOv5): {yolov8_metrics['Precision'] - yolov5_metrics['Precision']:.4f}")
print(f"Recall Difference (YOLOv8 - YOLOv5): {yolov8_metrics['Recall'] - yolov5_metrics['Recall']:.4f}")
print(f"FPS Difference (YOLOv8 - YOLOv5): {yolov8_fps - yolov5_fps:.2f}")
print(f"Model Size Difference (YOLOv8 - YOLOv5): {yolov8_size - yolov5_size:.2f} MB")


=== Performance Differences ===
mAP@0.5 Difference (YOLOv8 - YOLOv5): 0.0028
mAP@0.5:0.95 Difference (YOLOv8 - YOLOv5): 0.0435
Precision Difference (YOLOv8 - YOLOv5): 0.0123
Recall Difference (YOLOv8 - YOLOv5): 0.0170
FPS Difference (YOLOv8 - YOLOv5): -2.92
Model Size Difference (YOLOv8 - YOLOv5): 67.71 MB


In [17]:
# Save results to CSV
output_path = 'model_evaluation_results.csv'
df_comparison.to_csv(output_path, index=False)
print(f"\nResults saved to {output_path}")


Results saved to model_evaluation_results.csv


In [18]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (18, 12)

# Create subplots
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('YOLOv5 vs YOLOv8: Accuracy & Resource Usage Comparison', fontsize=16, fontweight='bold')

models = df_comparison['Model'].tolist()
colors = ['#FF6B6B', '#4ECDC4']

# ===== ACCURACY METRICS =====
# mAP@0.5
ax = axes[0, 0]
ax.bar(models, df_comparison['mAP@0.5'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Score', fontsize=10, fontweight='bold')
ax.set_title('mAP@0.5', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(df_comparison['mAP@0.5']) * 1.2)
for i, v in enumerate(df_comparison['mAP@0.5']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# mAP@0.5:0.95
ax = axes[0, 1]
ax.bar(models, df_comparison['mAP@0.5:0.95'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Score', fontsize=10, fontweight='bold')
ax.set_title('mAP@0.5:0.95', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(df_comparison['mAP@0.5:0.95']) * 1.2)
for i, v in enumerate(df_comparison['mAP@0.5:0.95']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# Precision
ax = axes[0, 2]
ax.bar(models, df_comparison['Precision'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Score', fontsize=10, fontweight='bold')
ax.set_title('Precision', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(df_comparison['Precision']) * 1.2)
for i, v in enumerate(df_comparison['Precision']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# Recall
ax = axes[1, 0]
ax.bar(models, df_comparison['Recall'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Score', fontsize=10, fontweight='bold')
ax.set_title('Recall', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(df_comparison['Recall']) * 1.2)
for i, v in enumerate(df_comparison['Recall']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# FPS
ax = axes[1, 1]
ax.bar(models, df_comparison['FPS'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('FPS', fontsize=10, fontweight='bold')
ax.set_title('Frames Per Second (FPS)', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(df_comparison['FPS']) * 1.2)
for i, v in enumerate(df_comparison['FPS']):
    ax.text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

# Model Size
ax = axes[1, 2]
ax.bar(models, df_comparison['Model Size (MB)'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Size (MB)', fontsize=10, fontweight='bold')
ax.set_title('Model File Size (MB)', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(df_comparison['Model Size (MB)']) * 1.2)
for i, v in enumerate(df_comparison['Model Size (MB)']):
    ax.text(i, v + 5, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

# ===== RESOURCE USAGE =====
# CPU Usage
ax = axes[2, 0]
ax.bar(models, df_comparison['CPU Usage (%)'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Usage (%)', fontsize=10, fontweight='bold')
ax.set_title('CPU Usage (%)', fontweight='bold', fontsize=11, color='#FF9800')
ax.set_ylim(0, 100)
for i, v in enumerate(df_comparison['CPU Usage (%)']):
    ax.text(i, v + 2, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=9)

# RAM Usage
ax = axes[2, 1]
ax.bar(models, df_comparison['RAM Usage (MB)'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Usage (MB)', fontsize=10, fontweight='bold')
ax.set_title('RAM Usage (MB)', fontweight='bold', fontsize=11, color='#2196F3')
max_ram = max(df_comparison['RAM Usage (MB)'])
ax.set_ylim(0, max_ram * 1.2)
for i, v in enumerate(df_comparison['RAM Usage (MB)']):
    ax.text(i, v + max_ram*0.05, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

# GPU Memory
ax = axes[2, 2]
ax.bar(models, df_comparison['GPU Memory (MB)'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Usage (MB)', fontsize=10, fontweight='bold')
ax.set_title('GPU Memory Usage (MB)', fontweight='bold', fontsize=11, color='#4CAF50')
max_gpu = max(df_comparison['GPU Memory (MB)'])
ax.set_ylim(0, max(max_gpu * 1.2, 100))
for i, v in enumerate(df_comparison['GPU Memory (MB)']):
    ax.text(i, v + max_gpu*0.05, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComparison chart saved as 'model_comparison.png'")

/var/folders/f9/3m3jbph106n72g4wt0z2r5pc0000gn/T/ipykernel_91803/1023955247.py:100: UserWarning: Tight layout not applied. tight_layout cannot make axes height small enough to accommodate all axes decorations.
  plt.tight_layout()


<Figure size 1800x1200 with 9 Axes>


Comparison chart saved as 'model_comparison.png'


In [19]:
# Restart kernel to clear old imports
import subprocess
subprocess.run(['pkill', '-f', 'jupyter'], check=False)
print("Kernel will be restarted. Re-run all cells.")

Kernel will be restarted. Re-run all cells.
